In [ ]:
import os, time
import numpy as np, pandas as pd
from sqlalchemy import create_engine

pg = f"postgresql+psycopg2://{os.getenv('PG_USER','postgres')}:{os.getenv('PG_PASSWORD','postgres')}@{os.getenv('PG_HOST','postgres')}:{os.getenv('PG_PORT','5432')}/{os.getenv('PG_DB','nyc_taxi')}"
eng = create_engine(pg)

q = """
SELECT service_type, pickup_datetime, pu_borough, pu_zone,
       passenger_count, trip_distance, fare_amount, tip_amount, total_amount
FROM analytics.obt_trips
WHERE year BETWEEN 2015 AND 2025
"""
df = pd.read_sql(q, eng)
df["year"] = df["pickup_datetime"].dt.year
df["month"] = df["pickup_datetime"].dt.month
df["pickup_hour"] = df["pickup_datetime"].dt.hour
df["pickup_dow"] = df["pickup_datetime"].dt.dayofweek

# Anti-leakage: target y features sólo de pickup/contexto
TARGET = "total_amount"
feats_num = ["trip_distance","passenger_count","pickup_hour","pickup_dow","month","year"]
feats_cat = ["service_type","pu_borough"]  # controla cardinalidad

# Split temporal (ajusta si tu set cambia)
train = df[df["year"]<=2019]
val   = df[(df["year"]>=2020) & (df["year"]<=2022)]
test  = df[df["year"]>=2023]

def XY(data):
    Xn = data[feats_num].copy()
    Xc = data[feats_cat].astype("category").copy()
    y  = data[TARGET].astype(float).values
    return Xn, Xc, y

Xn_tr, Xc_tr, y_tr = XY(train)
Xn_va, Xc_va, y_va = XY(val)
Xn_te, Xc_te, y_te = XY(test)
len(train), len(val), len(test)

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

num_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, include_bias=False))  # si explota dim, quítalo o limita features
])

cat_pipe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

prepro = ColumnTransformer([
    ("num", num_pipe, feats_num),
    ("cat", cat_pipe, feats_cat)
])

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet, SGDRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

models = {
    "Ridge": Ridge(alpha=1.0, random_state=42),
    "Lasso": Lasso(alpha=0.001, random_state=42, max_iter=2000),
    "ElasticNet": ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=42, max_iter=2000),
    "SGDRegressor": SGDRegressor(random_state=42, max_iter=2000, learning_rate="invscaling")
}

def fit_eval(name, est):
    pipe = Pipeline([("prep", prepro), ("est", est)])
    t0 = time.time(); pipe.fit(train[feats_num+feats_cat], y_tr); t1=time.time()
    def metrics(X, y):
        p = pipe.predict(X)
        return dict(
            RMSE=mean_squared_error(y,p,squared=False),
            MAE=mean_absolute_error(y,p),
            R2=r2_score(y,p)
        )
    m_val = metrics(val[feats_num+feats_cat], y_va)
    m_tes = metrics(test[feats_num+feats_cat], y_te)
    return {"model":name,"time_s":t1-t0,"val":m_val,"test":m_tes,"pipe":pipe}

results = [fit_eval(n,m) for n,m in models.items()]
pd.DataFrame([{**{"model":r["model"],"time_s":round(r["time_s"],2)},
               **{f"val_{k}":v for k,v in r["val"].items()},
               **{f"test_{k}":v for k,v in r["test"].items()}} for r in results]).sort_values("val_RMSE")

In [ ]:
# Mini implementación ilustrativa (no ultra-optimizada)
import numpy as np

def ridge_gd(X, y, alpha=0.01, l2=1.0, iters=2000):
    Xb = np.c_[np.ones((X.shape[0],1)), X]   # bias
    w  = np.zeros(Xb.shape[1])
    n  = len(y)
    for _ in range(iters):
        grad = (2/n) * Xb.T @ (Xb @ w - y) + 2*l2*np.r_[0,w[1:]]  # no regulariza bias
        w -= alpha*grad
    return w

# Usa el prepro para obtener matriz numérica
prep = prepro.fit(train[feats_num+feats_cat], y_tr)
X_tr = prep.transform(train[feats_num+feats_cat])
X_va = prep.transform(val[feats_num+feats_cat])
X_te = prep.transform(test[feats_num+feats_cat])

w = ridge_gd(X_tr, y_tr, alpha=0.001, l2=1.0, iters=2000)

def preds(X): return np.c_[np.ones((X.shape[0],1)), X] @ w
def metrics_np(y, p):
    rmse = np.sqrt(np.mean((y-p)**2)); mae = np.mean(np.abs(y-p))
    r2 = 1 - np.sum((y-p)**2)/np.sum((y-y.mean())**2)
    return rmse, mae, r2

pv = preds(X_va); pt = preds(X_te)
metrics_np(y_va, pv), metrics_np(y_te, pt)